# Session 3: Available View Types

Vitessce ships with a large number of interactive visualization and control views. In this notebook, we look at some types of views that you may not be familiar with.


The views available in Vitessce can generally be categorized into:
- Spatial view (covered in notebook 01)
- Scatterplot view for dimensionality reductions / embeddings
- Heatmap for expression data
- Genomic profiles for quantitative features aggregated by cell type
- Control views
  - Cell set manager
  - Feature list
  - Spatial layer controller
- Statistical plots
  - Cell set sizes bar plot
  - Expression distribution per cell set violin plots
  - Expression distribution histogram
  - Cell set and sample set sizes treemap
  - Volcano plot
  - Cell type composition bar plot
  - Gene set enrichment bar plot
  - Gating scatterplot

We cover:

1. **Embedding views** — UMAP, PCA, t-SNE scatterplots
2. **Expression views** — heatmap, dot plot, violin/distribution plot
3. **Spatial views** — tissue image, layer controller
4. **Control views** — cell-set tree, gene list, status bar

We use the EasyVitessce Scanpy API for most examples because it is the most concise, then show how to access additional view types through the lower-level `VitessceConfig` API.

In [ ]:
!pip install "vitessce[all]==3.9.2" "scanpy" "easy-vitessce==0.0.12" "spatialdata==0.7.3" "spatialdata-plot==0.4.0"

In [ ]:
import easy_vitessce as ev
import scanpy as sc

# Load the reduced PBMC dataset (ships with Scanpy, no download required)
adata = sc.datasets.pbmc68k_reduced()
adata

In [ ]:
# Exercise 2 — interactive dot plot
sc.pl.dotplot(
    adata,
    var_names=["C1QA", "PSAP", "CD79A", "CD79B", "CST3", "LYZ", "ANXA1", "S100A4"],
    groupby="bulk_labels",
)

In [ ]:
# Exercise 1 — your answer here
vc_ex = VitessceConfig(schema_version="1.0.15", name="Exercise 1")
ds_ex = vc_ex.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap_ex   = vc_ex.add_view(vt.SCATTERPLOT, dataset=ds_ex, mapping="UMAP")
violin_ex = vc_ex.add_view(vt.OBS_SET_FEATURE_VALUE_DISTRIBUTION, dataset=ds_ex)
sets_ex   = vc_ex.add_view(vt.OBS_SETS, dataset=ds_ex)

vc_ex.layout(???)   # TODO: arrange the three views
vc_ex.widget()

## Exercise

### Exercise 1 — Add a violin plot to the PBMC dashboard

👉 **Starting from the `vc_hm` configuration** (the heatmap + UMAP example), add an `OBS_SET_FEATURE_VALUE_DISTRIBUTION` view showing violin plots.

Steps:
1. Add the violin view with `vc_hm.add_view(vt.OBS_SET_FEATURE_VALUE_DISTRIBUTION, dataset=ds_hm)`
2. Include it in the layout, e.g., `(umap_hm | obs_sets3) / (heatmap | violin)`
3. Re-run `vc_hm.widget()` to see the result

```python
# Template — fill in the blanks
vc_ex = VitessceConfig(schema_version="1.0.15", name="Exercise 1")
ds_ex = vc_ex.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap_ex   = vc_ex.add_view(vt.SCATTERPLOT, dataset=ds_ex, mapping="UMAP")
violin_ex = vc_ex.add_view(???, dataset=ds_ex)        # Add the violin view
sets_ex   = vc_ex.add_view(vt.OBS_SETS, dataset=ds_ex)

vc_ex.layout(???)   # Arrange the views
vc_ex.widget()
```

### Exercise 2 — Explore the dot plot interactively

The dot plot view in Vitessce is interactive: you can click gene names to see them highlighted in other views. Run the cell below and try clicking a gene name in the dot plot row labels.

```python
sc.pl.dotplot(
    adata,
    var_names=["C1QA", "PSAP", "CD79A", "CD79B", "CST3", "LYZ", "ANXA1", "S100A4"],
    groupby="bulk_labels",
)
```

**What to notice:**
- Hovering over a dot shows the exact mean expression and percentage
- Clicking a gene selects it as the active feature (visible in other linked views if you have a multi-view layout)

In [ ]:
vc_ctrl = VitessceConfig(schema_version="1.0.15", name="Control Views Demo")
ds_ctrl = vc_ctrl.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap_c  = vc_ctrl.add_view(vt.SCATTERPLOT, dataset=ds_ctrl, mapping="UMAP")
sets_c  = vc_ctrl.add_view(vt.OBS_SETS, dataset=ds_ctrl)
genes_c = vc_ctrl.add_view(vt.FEATURE_LIST, dataset=ds_ctrl)
status  = vc_ctrl.add_view(vt.STATUS, dataset=ds_ctrl)

# Status bar is typically compact; place it below the main content
vc_ctrl.layout((umap_c | (sets_c / genes_c)) / status)
vc_ctrl.widget()

## 4. Control views

Control views don't render data directly but provide UI panels that let users interact with the visualization.

| View type | Purpose |
|---|---|
| `vt.OBS_SETS` | Navigate cell-type hierarchy; select/deselect groups |
| `vt.FEATURE_LIST` | Search for genes; click to color embeddings by expression |
| `vt.STATUS` | Shows hover information — gene name, cell ID, expression value |
| `"layerControllerBeta"` | Toggle image channels on/off, adjust contrast windows and colors |

The example below adds a `STATUS` view below the UMAP. Hover over points in the scatterplot to see live cell information in the status bar.

In [ ]:
import os, zipfile
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import spatialdata_plot  # noqa: F401

data_dir = "data"
sdata_filepath = join(data_dir, "visium.spatialdata.zarr")

if not isdir(sdata_filepath):
    zip_filepath = join(data_dir, "visium.spatialdata.zarr.zip")
    if not isfile(zip_filepath):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve("https://data-2.vitessce.io/sdata-datasets/visium.spatialdata.zarr.zip", zip_filepath)
    with zipfile.ZipFile(zip_filepath, "r") as z:
        z.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_filepath)

from spatialdata import read_zarr
sdata = read_zarr(sdata_filepath)

# Interactive spatial view colored by Fth1 expression
sdata.pl.render_images("ST8059050_hires_image") \
    .pl.render_shapes("ST8059050", color="Fth1", table_name="table") \
    .pl.show("ST8059050")

## 3. Spatial views

### 3a. The spatial viewer (`spatialBeta`)

The spatial viewer renders the tissue image and overlays data layers such as Visium spots colored by gene expression. Use `.pl.render_images()` and `.pl.render_shapes()` with EasyVitessce to compose the spatial view from a SpatialData object.

In [ ]:
import os
from os.path import join, isdir
from vitessce import VitessceConfig, ViewType as vt, AnnDataWrapper
from vitessce.data_utils import optimize_adata, VAR_CHUNK_SIZE

zarr_path = join("data", "pbmc68k.zarr")
if not isdir(zarr_path):
    os.makedirs("data", exist_ok=True)
    adata_opt = optimize_adata(
        adata,
        obs_cols=["bulk_labels", "louvain"],
        obsm_keys=["X_umap", "X_pca"],
        optimize_X=True,
    )
    adata_opt.write_zarr(zarr_path, chunks=[adata_opt.shape[0], VAR_CHUNK_SIZE])

vc_hm = VitessceConfig(schema_version="1.0.15", name="Heatmap + UMAP")
ds_hm = vc_hm.add_dataset(name="PBMC").add_object(AnnDataWrapper(
    adata_store=zarr_path,
    obs_set_paths=["obs/bulk_labels"],
    obs_set_names=["Cell Type"],
    obs_embedding_paths=["obsm/X_umap"],
    obs_embedding_names=["UMAP"],
    obs_feature_matrix_path="X",
))

umap_hm   = vc_hm.add_view(vt.SCATTERPLOT, dataset=ds_hm, mapping="UMAP")
heatmap   = vc_hm.add_view(vt.HEATMAP, dataset=ds_hm)
obs_sets3 = vc_hm.add_view(vt.OBS_SETS, dataset=ds_hm)

vc_hm.layout((umap_hm | obs_sets3) / heatmap)
vc_hm.widget()

### 2c. Heatmap (`heatmap`)

A **heatmap** renders a cells × genes expression matrix where color intensity encodes the expression level. Use it to compare expression patterns across many genes simultaneously and to validate cluster markers.

The EasyVitessce heatmap is accessible via the low-level `VitessceConfig` API. Let's build a compact example.

In [ ]:
sc.pl.violin(adata, keys="LYZ", groupby="bulk_labels")

### 2b. Violin / distribution plot (`obsSetFeatureValueDistribution`)

A **violin plot** shows the full distribution of expression values for one gene across cell-type groups. It combines a box plot (median, quartiles) with a density estimate (the width of the violin shape).

Use violin plots to understand whether expression differences between groups are large or variable.

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=["C1QA", "PSAP", "CD79A", "CD79B", "CST3", "LYZ"],
    groupby="bulk_labels",
)

## 2. Expression views

### 2a. Dot plot (`dotPlot`)

A **dot plot** encodes two quantities simultaneously for each gene × cell-type combination:

- **Dot size** — fraction of cells in the group that express the gene (above a threshold)
- **Dot color** — mean expression level across all cells in the group

Dot plots are useful for quickly comparing marker gene expression across many cell types.

In [ ]:
# PCA colored by a gene — try changing CD79A to another gene name
sc.pl.pca(adata, color="CD79A")

In [ ]:
# UMAP colored by cell type label
sc.pl.embedding(adata, basis="umap", color="bulk_labels")

## 1. Embedding views (`scatterplot`)

A **scatterplot** view renders a 2D dimensionality reduction such as UMAP or PCA. Each point is one cell. Points can be colored by:

- **Cell type / cluster label** — categorical coloring
- **Gene expression** — continuous colormap from low (gray) to high (color)

EasyVitessce hooks into `sc.pl.embedding()`. Pass `basis=` to choose the embedding and `color=` to choose what to encode with color.